# DeepTrace: A Hybrid Transformer-Ensemble Framework for Malicious URL Detection

**Beyond Lexical Features: Leveraging Transformer Embeddings and Tree-Based Leaf Encoding for URL Threat Classification**

Master's Dissertation Implementation Notebook (Final Scope)

This notebook implements the complete, final-scope experimental pipeline: data cleaning, binary labeling, balanced sampling, BERT fine-tuning, CLS embedding extraction, classical classifiers (XGBoost/Random Forest) on BERT embeddings, handcrafted URL feature extraction, hybrid feature construction, the proposed DeepTrace model (Hybrid -> XGBoost leaf-encoding -> Random Forest), hyperparameter tuning, the final tuned DeepTrace model, a 6-model comparison, and error analysis.

Note: Support Vector Machine (SVM) classifiers were evaluated during earlier research but are excluded from this final notebook, since DeepTrace's architecture does not use SVM and the final paper reports only the six models directly relevant to the proposed pipeline's development and comparison.

Run all cells sequentially from a fresh Colab runtime with GPU enabled (Runtime > Change runtime type > GPU).

## Section 1: Verify GPU

In [ ]:
import torch

print("PyTorch Version :", torch.__version__)
print("CUDA Available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Device      :", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. BERT fine-tuning will be slow.")
    print("Go to Runtime > Change runtime type > select GPU (T4/L4/A100).")

## Section 2: Install Required Packages

In [ ]:
!pip install -q scikit-learn xgboost joblib seaborn

## Section 3: Import Core Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Core libraries loaded successfully.")
print("Random seed set to:", RANDOM_SEED)

## Section 4: Upload Dataset

In [ ]:
from google.colab import files

print("Please select malicious_phish.csv from your computer...")
uploaded = files.upload()

DATA_PATH = list(uploaded.keys())[0]
print("File uploaded:", DATA_PATH)

## Section 5: Load Dataset

In [ ]:
df_raw = pd.read_csv(DATA_PATH)

print("Raw dataset shape:", df_raw.shape)
df_raw.head()

## Section 6: Dataset Structure Check

In [ ]:
print(df_raw.info())
print()
print("Columns:", list(df_raw.columns))

## Section 7: Missing Values Check

In [ ]:
missing_counts = df_raw.isnull().sum()
print("Missing values per column:")
print(missing_counts)

total_missing = missing_counts.sum()
print(f"\nTotal missing values: {total_missing}")

## Section 8: Original Class Distribution (Before Cleaning)

In [ ]:
print("Original class distribution:")
print(df_raw["type"].value_counts())

plt.figure(figsize=(7,5))
sns.countplot(x=df_raw["type"], order=df_raw["type"].value_counts().index)
plt.title("Original Class Distribution (Raw Dataset)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("Figure0_Raw_Class_Distribution.png", dpi=300)
plt.show()

## Section 9: Data Cleaning — Remove Missing Values and Duplicate URLs

In [ ]:
df_clean = df_raw.dropna(subset=["url", "type"]).copy()
print("Shape after dropping missing values:", df_clean.shape)

before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["url"]).reset_index(drop=True)
after_dedup = len(df_clean)

print(f"Shape after removing duplicate URLs: {df_clean.shape}")
print(f"Duplicates removed: {before_dedup - after_dedup}")

## Section 10: Cleaned Class Distribution

In [ ]:
print("Cleaned class distribution:")
print(df_clean["type"].value_counts())

plt.figure(figsize=(7,5))
sns.countplot(x=df_clean["type"], order=df_clean["type"].value_counts().index)
plt.title("Class Distribution After Cleaning (Duplicates Removed)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("Figure0b_Cleaned_Class_Distribution.png", dpi=300)
plt.show()

## Section 11: Save Cleaned Dataset

In [ ]:
df_clean.to_csv("Cleaned_Malicious_URL_Dataset.csv", index=False)
print("Cleaned dataset saved as: Cleaned_Malicious_URL_Dataset.csv")
print("Final cleaned shape:", df_clean.shape)

## Section 12: Binary Label Conversion

Benign -> Safe (0); Phishing, Malware, Defacement -> Unsafe (1)

In [ ]:
safe_df = df_clean[df_clean["type"] == "benign"].copy()

unsafe_df = df_clean[
    df_clean["type"].isin(["phishing", "malware", "defacement"])
].copy()

print("Safe (Benign) samples   :", len(safe_df))
print("Unsafe (Phish/Mal/Def) samples:", len(unsafe_df))

## Section 13: Balanced 50K Sampling (25,000 Safe + 25,000 Unsafe, seed=42)

In [ ]:
safe_sample = safe_df.sample(n=25000, random_state=RANDOM_SEED)
unsafe_sample = unsafe_df.sample(n=25000, random_state=RANDOM_SEED)

safe_sample["label"] = 0
unsafe_sample["label"] = 1

binary_df = pd.concat([safe_sample, unsafe_sample])
binary_df = binary_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print("Balanced dataset shape:", binary_df.shape)
binary_df.head()

## Section 14: Verify Balance and Save

In [ ]:
print("Label distribution:")
print(binary_df["label"].value_counts())

plt.figure(figsize=(6,5))
sns.countplot(x=binary_df["label"])
plt.title("Safe (0) vs Unsafe (1) - Balanced 50K Dataset")
plt.xlabel("Label")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("Figure0c_Balanced_Distribution.png", dpi=300)
plt.show()

binary_df.to_csv("Balanced_50K_URL_Dataset.csv", index=False)
print("Saved: Balanced_50K_URL_Dataset.csv")

## Section 15: Train/Test Split (80/20, Stratified, seed=42)

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    binary_df["url"],
    binary_df["label"],
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=binary_df["label"]
)

print("Train samples:", len(train_texts))
print("Test samples :", len(test_texts))

## Section 16: Save Train/Test Split

In [ ]:
import joblib

train_df_out = pd.DataFrame({"url": train_texts, "label": train_labels})
test_df_out  = pd.DataFrame({"url": test_texts,  "label": test_labels})

train_df_out.to_csv("Train_40K.csv", index=False)
test_df_out.to_csv("Test_10K.csv", index=False)

joblib.dump(
    (train_texts, test_texts, train_labels, test_labels),
    "binary_train_test_split.pkl"
)

print("Saved: Train_40K.csv, Test_10K.csv, binary_train_test_split.pkl")
print("Train label distribution:")
print(train_labels.value_counts())
print("Test label distribution:")
print(test_labels.value_counts())

## Section 17: Install Transformers Stack

In [ ]:
!pip install -q transformers datasets accelerate evaluate

## Section 18: Imports for BERT Fine-Tuning

In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

torch.manual_seed(RANDOM_SEED)

print("Transformers stack imported. Seed set to", RANDOM_SEED)

## Section 19: Build Hugging Face Datasets

In [ ]:
train_df = pd.DataFrame({
    "text": train_texts.tolist(),
    "label": train_labels.tolist()
})
test_df = pd.DataFrame({
    "text": test_texts.tolist(),
    "label": test_labels.tolist()
})

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(test_dataset)

## Section 20: Load BERT Tokenizer

In [ ]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded:", MODEL_NAME)

## Section 21: Tokenize Datasets (max_length=128)

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenization complete.")
print(tokenized_train)

## Section 22: Load BERT Model for Sequence Classification

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print("Model loaded:", MODEL_NAME)

## Section 23: Define Evaluation Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted"
    )
    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

## Section 24: Training Arguments (Frozen Hyperparameters)

Seed=42, Epochs=2, Learning Rate=2e-5, Batch Size=16, Weight Decay=0.01, Max Length=128

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_binary_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    seed=RANDOM_SEED,
    fp16=True,
    logging_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer configured.")

## Section 25: Train BERT

In [ ]:
train_result = trainer.train()
print(train_result)

## Section 26: Evaluate on Test Set

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

## Section 27: Generate Predictions on Test Set

In [ ]:
predictions_output = trainer.predict(tokenized_test)

y_true = predictions_output.label_ids
y_pred = np.argmax(predictions_output.predictions, axis=1)

from scipy.special import softmax
y_probs = softmax(predictions_output.predictions, axis=1)[:, 1]

print("Predictions generated.")
print("Shapes -> y_true:", y_true.shape, "y_pred:", y_pred.shape, "y_probs:", y_probs.shape)

## Section 28: Classification Report

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    y_true, y_pred, target_names=["Safe", "Unsafe"]
)
print(report)

with open("BERT_Classification_Report.txt", "w") as f:
    f.write(report)

## Section 29: Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

cm_bert = confusion_matrix(y_true, y_pred)
print(cm_bert)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm_bert, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Safe","Unsafe"], yticklabels=["Safe","Unsafe"]
)
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("BERT Confusion Matrix (Binary URL Classification)")
plt.tight_layout()
plt.savefig("Figure12_BERT_Confusion_Matrix.png", dpi=300)
plt.show()

## Section 30: ROC Curve

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_true, y_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, linewidth=2, label=f"BERT (AUC = {roc_auc:.4f})")
plt.plot([0,1],[0,1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("BERT ROC Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("Figure13_BERT_ROC_Curve.png", dpi=300)
plt.show()

print("BERT ROC-AUC:", roc_auc)

## Section 31: Precision-Recall Curve

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

prec_curve, rec_curve, _ = precision_recall_curve(y_true, y_probs)
avg_prec = average_precision_score(y_true, y_probs)

plt.figure(figsize=(6,6))
plt.plot(rec_curve, prec_curve, linewidth=2, label=f"BERT (AP = {avg_prec:.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("BERT Precision-Recall Curve")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("Figure14_BERT_PR_Curve.png", dpi=300)
plt.show()

print("BERT Average Precision:", avg_prec)

## Section 32: Learning Curve (Training/Validation Loss)

In [ ]:
log_history = trainer.state.log_history

train_epochs, train_losses = [], []
eval_epochs, eval_losses = [], []

for entry in log_history:
    if "loss" in entry and "epoch" in entry:
        train_epochs.append(entry["epoch"])
        train_losses.append(entry["loss"])
    if "eval_loss" in entry and "epoch" in entry:
        eval_epochs.append(entry["epoch"])
        eval_losses.append(entry["eval_loss"])

plt.figure(figsize=(8,6))
plt.plot(train_epochs, train_losses, marker="o", label="Training Loss")
plt.plot(eval_epochs, eval_losses, marker="s", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("BERT Training vs Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("Figure15_BERT_Learning_Curve.png", dpi=300)
plt.show()

history_df = pd.DataFrame(log_history)
history_df.to_csv("BERT_Training_History.csv", index=False)
print("Training history saved.")

## Section 33: Save Model, Tokenizer, and Evaluation CSV

In [ ]:
eval_summary = pd.DataFrame([{
    "Model": "BERT (bert-base-uncased)",
    "Accuracy": eval_results["eval_accuracy"],
    "Precision": eval_results["eval_precision"],
    "Recall": eval_results["eval_recall"],
    "F1": eval_results["eval_f1"],
    "ROC_AUC": roc_auc,
    "Average_Precision": avg_prec
}])
eval_summary.to_csv("BERT_Evaluation_Summary.csv", index=False)

BEST_MODEL_DIR = "bert_binary_best_model"
model.save_pretrained(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("Saved best model + tokenizer to:", BEST_MODEL_DIR)
print(eval_summary)

## Section 34: Reload Best BERT Model (Embedding Mode)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

embed_tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)
embed_model = AutoModel.from_pretrained(BEST_MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embed_model.to(device)
embed_model.eval()

print("Model reloaded for embedding extraction on:", device)

## Section 35: CLS Embedding Extraction Function

In [ ]:
from tqdm import tqdm
import numpy as np

def generate_embeddings(texts, batch_size=32, max_length=128):
    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch = [str(t) for t in texts[i:i+batch_size]]
            inputs = embed_tokenizer(
                batch,
                padding="max_length",
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = embed_model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
    return np.vstack(embeddings)

print("Embedding function ready (batched for speed).")

## Section 36: Extract Train Embeddings

In [ ]:
X_train_bert = generate_embeddings(train_texts.tolist())
print("X_train_bert shape:", X_train_bert.shape)

## Section 37: Extract Test Embeddings

In [ ]:
X_test_bert = generate_embeddings(test_texts.tolist())
print("X_test_bert shape:", X_test_bert.shape)

## Section 38: Save Embeddings and Labels

In [ ]:
np.save("X_train_bert_binary.npy", X_train_bert)
np.save("X_test_bert_binary.npy", X_test_bert)
np.save("train_labels_binary.npy", np.array(train_labels))
np.save("test_labels_binary.npy", np.array(test_labels))

print("Saved:")
print(" - X_train_bert_binary.npy", X_train_bert.shape)
print(" - X_test_bert_binary.npy ", X_test_bert.shape)
print(" - train_labels_binary.npy", len(train_labels))
print(" - test_labels_binary.npy ", len(test_labels))

## Section 39: Imports for Classical ML Experiments (XGBoost, Random Forest)

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

train_labels_arr = np.load("train_labels_binary.npy")
test_labels_arr = np.load("test_labels_binary.npy")

print("Setup complete. Train/test label arrays loaded.")

## Section 40: BERT Embeddings + XGBoost

In [ ]:
xgb_bert = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_SEED, eval_metric="logloss"
)
xgb_bert.fit(X_train_bert, train_labels_arr)

xgb_bert_preds = xgb_bert.predict(X_test_bert)
xgb_bert_probs = xgb_bert.predict_proba(X_test_bert)[:, 1]

xgb_bert_metrics = {
    "Accuracy": accuracy_score(test_labels_arr, xgb_bert_preds),
    "Precision": precision_score(test_labels_arr, xgb_bert_preds),
    "Recall": recall_score(test_labels_arr, xgb_bert_preds),
    "F1": f1_score(test_labels_arr, xgb_bert_preds)
}
print("BERT + XGBoost:", xgb_bert_metrics)
print(classification_report(test_labels_arr, xgb_bert_preds, target_names=["Safe","Unsafe"]))

cm_xgb_bert = confusion_matrix(test_labels_arr, xgb_bert_preds)
print(cm_xgb_bert)

np.save("bert_xgb_predictions.npy", xgb_bert_preds)
np.save("bert_xgb_probs.npy", xgb_bert_probs)

## Section 41: BERT Embeddings + Random Forest

In [ ]:
rf_bert = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)
rf_bert.fit(X_train_bert, train_labels_arr)

rf_bert_preds = rf_bert.predict(X_test_bert)
rf_bert_probs = rf_bert.predict_proba(X_test_bert)[:, 1]

rf_bert_metrics = {
    "Accuracy": accuracy_score(test_labels_arr, rf_bert_preds),
    "Precision": precision_score(test_labels_arr, rf_bert_preds),
    "Recall": recall_score(test_labels_arr, rf_bert_preds),
    "F1": f1_score(test_labels_arr, rf_bert_preds)
}
print("BERT + Random Forest:", rf_bert_metrics)
print(classification_report(test_labels_arr, rf_bert_preds, target_names=["Safe","Unsafe"]))

cm_rf_bert = confusion_matrix(test_labels_arr, rf_bert_preds)
print(cm_rf_bert)

np.save("bert_rf_predictions.npy", rf_bert_preds)
np.save("bert_rf_probs.npy", rf_bert_probs)

## Section 42: Figure — Confusion Matrices (BERT + XGBoost, BERT + Random Forest)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))

for ax, cm, title in zip(
    axes,
    [cm_xgb_bert, cm_rf_bert],
    ["BERT + XGBoost", "BERT + Random Forest"]
):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Safe","Unsafe"], yticklabels=["Safe","Unsafe"], ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig("Figure16_BERT_Classifiers_Confusion_Matrices.png", dpi=300)
plt.show()

## Section 43: Figure — ROC Curve Comparison (BERT + Classifiers)

In [ ]:
plt.figure(figsize=(7,7))

for probs, label in zip(
    [xgb_bert_probs, rf_bert_probs],
    ["BERT+XGBoost", "BERT+RF"]
):
    fpr, tpr, _ = roc_curve(test_labels_arr, probs)
    roc_auc_val = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f"{label} (AUC={roc_auc_val:.4f})")

plt.plot([0,1],[0,1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison - BERT Embeddings + Classical Classifiers")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("Figure17_BERT_Classifiers_ROC_Comparison.png", dpi=300)
plt.show()

## Section 44: Handcrafted URL Feature Extraction Functions

In [ ]:
from urllib.parse import urlparse
import math
from collections import Counter

def calculate_entropy(text):
    counter = Counter(text)
    length = len(text)
    entropy = 0
    for count in counter.values():
        probability = count / length
        entropy -= probability * math.log2(probability)
    return entropy

def extract_url_features(url):
    url = str(url)
    parsed = urlparse(url)
    domain = parsed.netloc
    path = parsed.path

    features = [
        len(url),
        len(domain),
        len(path),
        url.count("."),
        url.count("-"),
        url.count("_"),
        url.count("/"),
        url.count("?"),
        url.count("="),
        sum(c.isdigit() for c in url),
        sum(c.isalpha() for c in url),
        url.count("@"),
        url.count("%"),
        url.count("&"),
        calculate_entropy(url),
        int("https" in url),
        int("login" in url.lower()),
        int("verify" in url.lower()),
        int("secure" in url.lower()),
        int("account" in url.lower())
    ]
    return features

print("Feature extraction function defined (20 handcrafted features).")

## Section 45: Extract Handcrafted Features

In [ ]:
X_train_url = np.array([extract_url_features(url) for url in train_texts])
X_test_url = np.array([extract_url_features(url) for url in test_texts])

print("X_train_url shape:", X_train_url.shape)
print("X_test_url shape :", X_test_url.shape)

## Section 46: Save Handcrafted Features

In [ ]:
np.save("X_train_url_features.npy", X_train_url)
np.save("X_test_url_features.npy", X_test_url)

feature_names = [
    "url_length", "domain_length", "path_length", "dot_count", "hyphen_count",
    "underscore_count", "slash_count", "question_count", "equals_count",
    "digit_count", "alpha_count", "at_count", "percent_count", "ampersand_count",
    "entropy", "https_flag", "login_keyword", "verify_keyword",
    "secure_keyword", "account_keyword"
]

train_url_df = pd.DataFrame(X_train_url, columns=feature_names)
print(train_url_df.describe())

## Section 47: Build Hybrid Feature Vectors (768 + 20 = 788)

In [ ]:
X_train_hybrid = np.concatenate([X_train_bert, X_train_url], axis=1)
X_test_hybrid = np.concatenate([X_test_bert, X_test_url], axis=1)

print("X_train_hybrid shape:", X_train_hybrid.shape)
print("X_test_hybrid shape :", X_test_hybrid.shape)

np.save("X_train_hybrid.npy", X_train_hybrid)
np.save("X_test_hybrid.npy", X_test_hybrid)
print("Hybrid feature vectors saved.")

## Section 48: Hybrid Features + XGBoost

In [ ]:
hybrid_xgb = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_SEED, eval_metric="logloss"
)
hybrid_xgb.fit(X_train_hybrid, train_labels_arr)

hybrid_xgb_preds = hybrid_xgb.predict(X_test_hybrid)
hybrid_xgb_probs = hybrid_xgb.predict_proba(X_test_hybrid)[:, 1]

hybrid_xgb_metrics = {
    "Accuracy": accuracy_score(test_labels_arr, hybrid_xgb_preds),
    "Precision": precision_score(test_labels_arr, hybrid_xgb_preds),
    "Recall": recall_score(test_labels_arr, hybrid_xgb_preds),
    "F1": f1_score(test_labels_arr, hybrid_xgb_preds)
}
print("Hybrid + XGBoost:", hybrid_xgb_metrics)
print(classification_report(test_labels_arr, hybrid_xgb_preds, target_names=["Safe","Unsafe"]))
cm_hybrid_xgb = confusion_matrix(test_labels_arr, hybrid_xgb_preds)
print(cm_hybrid_xgb)

np.save("hybrid_xgb_predictions.npy", hybrid_xgb_preds)
np.save("hybrid_xgb_probs.npy", hybrid_xgb_probs)

## Section 49: Hybrid Features + Random Forest

In [ ]:
hybrid_rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)
hybrid_rf.fit(X_train_hybrid, train_labels_arr)

hybrid_rf_preds = hybrid_rf.predict(X_test_hybrid)
hybrid_rf_probs = hybrid_rf.predict_proba(X_test_hybrid)[:, 1]

hybrid_rf_metrics = {
    "Accuracy": accuracy_score(test_labels_arr, hybrid_rf_preds),
    "Precision": precision_score(test_labels_arr, hybrid_rf_preds),
    "Recall": recall_score(test_labels_arr, hybrid_rf_preds),
    "F1": f1_score(test_labels_arr, hybrid_rf_preds)
}
print("Hybrid + Random Forest:", hybrid_rf_metrics)
print(classification_report(test_labels_arr, hybrid_rf_preds, target_names=["Safe","Unsafe"]))
cm_hybrid_rf = confusion_matrix(test_labels_arr, hybrid_rf_preds)
print(cm_hybrid_rf)

np.save("hybrid_rf_predictions.npy", hybrid_rf_preds)
np.save("hybrid_rf_probs.npy", hybrid_rf_probs)

## Section 50: Figure — Confusion Matrices (Hybrid + XGBoost, Hybrid + Random Forest)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))

for ax, cm, title in zip(
    axes,
    [cm_hybrid_xgb, cm_hybrid_rf],
    ["Hybrid + XGBoost", "Hybrid + Random Forest"]
):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
                xticklabels=["Safe","Unsafe"], yticklabels=["Safe","Unsafe"], ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig("Figure18_Hybrid_Classifiers_Confusion_Matrices.png", dpi=300)
plt.show()

## Section 51: Figure — ROC Curve Comparison (Hybrid + Classifiers)

In [ ]:
plt.figure(figsize=(7,7))
for probs, label in zip(
    [hybrid_xgb_probs, hybrid_rf_probs],
    ["Hybrid+XGBoost", "Hybrid+RF"]
):
    fpr, tpr, _ = roc_curve(test_labels_arr, probs)
    roc_auc_val = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f"{label} (AUC={roc_auc_val:.4f})")

plt.plot([0,1],[0,1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison - Hybrid Features + Classical Classifiers")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("Figure19_Hybrid_Classifiers_ROC_Comparison.png", dpi=300)
plt.show()

## Section 52: DeepTrace — Step 1: Train XGBoost (Leaf Generator)

**DeepTrace**: Hybrid features -> XGBoost -> leaf-node indices -> concatenate with hybrid features -> final Random Forest classifier.

In [ ]:
xgb_leaf_model = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_SEED, eval_metric="logloss"
)
xgb_leaf_model.fit(X_train_hybrid, train_labels_arr)

print("XGBoost leaf-generating model trained.")

## Section 53: DeepTrace — Step 2: Extract Leaf-Node Indices

In [ ]:
train_leaf_indices = xgb_leaf_model.apply(X_train_hybrid)
test_leaf_indices = xgb_leaf_model.apply(X_test_hybrid)

print("Train leaf indices shape:", train_leaf_indices.shape)
print("Test leaf indices shape :", test_leaf_indices.shape)

## Section 54: DeepTrace — Step 3: Build Final Feature Set (788 + 200 = 988-dim)

In [ ]:
X_train_final = np.concatenate([X_train_hybrid, train_leaf_indices], axis=1)
X_test_final = np.concatenate([X_test_hybrid, test_leaf_indices], axis=1)

print("X_train_final shape:", X_train_final.shape)
print("X_test_final shape :", X_test_final.shape)

np.save("X_train_final_deeptrace.npy", X_train_final)
np.save("X_test_final_deeptrace.npy", X_test_final)

## Section 55: DeepTrace — Step 4: Train Final Random Forest (Default Hyperparameters)

In [ ]:
deeptrace_rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)
deeptrace_rf.fit(X_train_final, train_labels_arr)

print("DeepTrace model (default hyperparameters) trained.")

## Section 56: DeepTrace — Step 5: Evaluation (Default Hyperparameters)

In [ ]:
deeptrace_preds = deeptrace_rf.predict(X_test_final)
deeptrace_probs = deeptrace_rf.predict_proba(X_test_final)[:, 1]

deeptrace_metrics = {
    "Accuracy": accuracy_score(test_labels_arr, deeptrace_preds),
    "Precision": precision_score(test_labels_arr, deeptrace_preds),
    "Recall": recall_score(test_labels_arr, deeptrace_preds),
    "F1": f1_score(test_labels_arr, deeptrace_preds)
}
print("DeepTrace (Default Hyperparameters):", deeptrace_metrics)
print(classification_report(test_labels_arr, deeptrace_preds, target_names=["Safe","Unsafe"]))

cm_deeptrace = confusion_matrix(test_labels_arr, deeptrace_preds)
print(cm_deeptrace)

np.save("deeptrace_default_predictions.npy", deeptrace_preds)
np.save("deeptrace_default_probs.npy", deeptrace_probs)

## Section 57: Figure — DeepTrace (Default) Confusion Matrix, ROC, PR Curve

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(cm_deeptrace, annot=True, fmt="d", cmap="Purples",
            xticklabels=["Safe","Unsafe"], yticklabels=["Safe","Unsafe"])
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("DeepTrace (Default Hyperparameters) - Confusion Matrix")
plt.tight_layout()
plt.savefig("Figure20_DeepTrace_Default_Confusion_Matrix.png", dpi=300)
plt.show()

fpr_p, tpr_p, _ = roc_curve(test_labels_arr, deeptrace_probs)
roc_auc_p = auc(fpr_p, tpr_p)

plt.figure(figsize=(6,6))
plt.plot(fpr_p, tpr_p, linewidth=2, label=f"DeepTrace Default (AUC={roc_auc_p:.4f})")
plt.plot([0,1],[0,1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("DeepTrace (Default Hyperparameters) - ROC Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("Figure21_DeepTrace_Default_ROC_Curve.png", dpi=300)
plt.show()

prec_p, rec_p, _ = precision_recall_curve(test_labels_arr, deeptrace_probs)
avg_prec_p = average_precision_score(test_labels_arr, deeptrace_probs)

plt.figure(figsize=(6,6))
plt.plot(rec_p, prec_p, linewidth=2, label=f"DeepTrace Default (AP={avg_prec_p:.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("DeepTrace (Default Hyperparameters) - Precision-Recall Curve")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("Figure22_DeepTrace_Default_PR_Curve.png", dpi=300)
plt.show()

print("DeepTrace (Default) ROC-AUC:", roc_auc_p)
print("DeepTrace (Default) Average Precision:", avg_prec_p)

## Section 58: Hyperparameter Tuning — XGBoost n_estimators

In [ ]:
import time

tuning_results_estimators = []
estimator_values = [50, 100, 150, 200, 250, 300, 400, 500]

for n in estimator_values:
    print(f"Training XGBoost with n_estimators={n} ...")
    start = time.time()

    xgb_tune = XGBClassifier(
        n_estimators=n, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_SEED, eval_metric="logloss"
    )
    xgb_tune.fit(X_train_hybrid, train_labels_arr)

    train_leaf = xgb_tune.apply(X_train_hybrid)
    test_leaf = xgb_tune.apply(X_test_hybrid)

    X_train_temp = np.concatenate([X_train_hybrid, train_leaf], axis=1)
    X_test_temp = np.concatenate([X_test_hybrid, test_leaf], axis=1)

    rf_tune = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)
    rf_tune.fit(X_train_temp, train_labels_arr)
    preds_tune = rf_tune.predict(X_test_temp)

    end = time.time()

    tuning_results_estimators.append({
        "Estimators": n,
        "Accuracy": accuracy_score(test_labels_arr, preds_tune),
        "Precision": precision_score(test_labels_arr, preds_tune),
        "Recall": recall_score(test_labels_arr, preds_tune),
        "F1": f1_score(test_labels_arr, preds_tune),
        "Training Time (s)": round(end - start, 2)
    })

results_estimators_df = pd.DataFrame(tuning_results_estimators)
results_estimators_df.to_csv("Tuning_XGBoost_Estimators.csv", index=False)
print(results_estimators_df)

## Section 59: Figure — XGBoost n_estimators Sensitivity

In [ ]:
plt.figure(figsize=(9,6))
plt.plot(results_estimators_df["Estimators"], results_estimators_df["F1"],
         marker="o", linewidth=2.5, markersize=8)
for x, y in zip(results_estimators_df["Estimators"], results_estimators_df["F1"]):
    plt.text(x, y + 0.00005, f"{y:.4f}", ha="center", fontsize=9)
plt.xlabel("Number of XGBoost Estimators")
plt.ylabel("F1 Score")
plt.title("Hyperparameter Sensitivity: XGBoost Number of Estimators")
plt.grid(linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("Figure23_XGBoost_Estimators_Tuning.png", dpi=300)
plt.show()

best_est = results_estimators_df.loc[results_estimators_df["F1"].idxmax()]
print("Best n_estimators configuration:")
print(best_est)

## Section 60: Hyperparameter Tuning — XGBoost max_depth

In [ ]:
tuning_results_depth = []
depth_values = [2, 3, 4, 5, 6, 7, 8, 10, 12]

BEST_N_ESTIMATORS = int(best_est["Estimators"])

for depth in depth_values:
    print(f"Training XGBoost with max_depth={depth} ...")
    start = time.time()

    xgb_tune = XGBClassifier(
        n_estimators=BEST_N_ESTIMATORS, max_depth=depth, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_SEED, eval_metric="logloss"
    )
    xgb_tune.fit(X_train_hybrid, train_labels_arr)

    train_leaf = xgb_tune.apply(X_train_hybrid)
    test_leaf = xgb_tune.apply(X_test_hybrid)

    X_train_temp = np.concatenate([X_train_hybrid, train_leaf], axis=1)
    X_test_temp = np.concatenate([X_test_hybrid, test_leaf], axis=1)

    rf_tune = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)
    rf_tune.fit(X_train_temp, train_labels_arr)
    preds_tune = rf_tune.predict(X_test_temp)

    end = time.time()

    tuning_results_depth.append({
        "Max Depth": depth,
        "Accuracy": accuracy_score(test_labels_arr, preds_tune),
        "Precision": precision_score(test_labels_arr, preds_tune),
        "Recall": recall_score(test_labels_arr, preds_tune),
        "F1": f1_score(test_labels_arr, preds_tune),
        "Training Time (s)": round(end - start, 2)
    })

results_depth_df = pd.DataFrame(tuning_results_depth)
results_depth_df.to_csv("Tuning_XGBoost_MaxDepth.csv", index=False)
print(results_depth_df)

## Section 61: Figure — XGBoost max_depth Sensitivity

In [ ]:
plt.figure(figsize=(9,6))
plt.plot(results_depth_df["Max Depth"], results_depth_df["F1"],
         marker="o", linewidth=2.5, markersize=8)
for x, y in zip(results_depth_df["Max Depth"], results_depth_df["F1"]):
    plt.text(x, y + 0.00005, f"{y:.4f}", ha="center", fontsize=9)
plt.xlabel("Maximum Tree Depth")
plt.ylabel("F1 Score")
plt.title("Hyperparameter Sensitivity: XGBoost Maximum Tree Depth")
plt.xticks(results_depth_df["Max Depth"])
plt.grid(linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("Figure24_XGBoost_MaxDepth_Tuning.png", dpi=300)
plt.show()

best_depth = results_depth_df.loc[results_depth_df["F1"].idxmax()]
print("Best max_depth configuration:")
print(best_depth)

## Section 62: Hyperparameter Tuning — Random Forest n_estimators

In [ ]:
tuning_results_rf = []
rf_estimator_values = [50, 100, 200, 300, 500, 700, 1000]

BEST_MAX_DEPTH = int(best_depth["Max Depth"])

xgb_optimized = XGBClassifier(
    n_estimators=BEST_N_ESTIMATORS, max_depth=BEST_MAX_DEPTH, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_SEED, eval_metric="logloss"
)
xgb_optimized.fit(X_train_hybrid, train_labels_arr)

train_leaf_opt = xgb_optimized.apply(X_train_hybrid)
test_leaf_opt = xgb_optimized.apply(X_test_hybrid)

X_train_opt = np.concatenate([X_train_hybrid, train_leaf_opt], axis=1)
X_test_opt = np.concatenate([X_test_hybrid, test_leaf_opt], axis=1)

print("Optimized XGBoost leaf features built. Shape:", X_train_opt.shape)

for n in rf_estimator_values:
    print(f"Training Random Forest with n_estimators={n} ...")
    start = time.time()

    rf_tune = RandomForestClassifier(n_estimators=n, random_state=RANDOM_SEED, n_jobs=-1)
    rf_tune.fit(X_train_opt, train_labels_arr)
    preds_tune = rf_tune.predict(X_test_opt)

    end = time.time()

    tuning_results_rf.append({
        "RF Estimators": n,
        "Accuracy": accuracy_score(test_labels_arr, preds_tune),
        "Precision": precision_score(test_labels_arr, preds_tune),
        "Recall": recall_score(test_labels_arr, preds_tune),
        "F1": f1_score(test_labels_arr, preds_tune),
        "Training Time (s)": round(end - start, 2)
    })

results_rf_df = pd.DataFrame(tuning_results_rf)
results_rf_df.to_csv("Tuning_RF_Estimators.csv", index=False)
print(results_rf_df)

## Section 63: Figure — Random Forest n_estimators Sensitivity

In [ ]:
plt.figure(figsize=(9,6))
plt.plot(results_rf_df["RF Estimators"], results_rf_df["F1"],
         marker="o", linewidth=2.5, markersize=8)
for x, y in zip(results_rf_df["RF Estimators"], results_rf_df["F1"]):
    plt.text(x, y + 0.00005, f"{y:.4f}", ha="center", fontsize=9)
plt.xlabel("Number of Random Forest Trees")
plt.ylabel("F1 Score")
plt.title("Hyperparameter Sensitivity: Random Forest Number of Estimators")
plt.grid(linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("Figure25_RF_Estimators_Tuning.png", dpi=300)
plt.show()

best_rf = results_rf_df.loc[results_rf_df["F1"].idxmax()]
print("Best RF n_estimators configuration:")
print(best_rf)

print("\n=== FINAL OPTIMAL CONFIGURATION ===")
print("XGBoost n_estimators:", BEST_N_ESTIMATORS)
print("XGBoost max_depth   :", BEST_MAX_DEPTH)
print("Random Forest n_estimators:", int(best_rf["RF Estimators"]))

## Section 64: Final DeepTrace — Step 1: XGBoost (Tuned)

In [ ]:
FINAL_XGB_ESTIMATORS = 150
FINAL_XGB_MAX_DEPTH = 10
FINAL_RF_ESTIMATORS = 200

xgb_final_optimized = XGBClassifier(
    n_estimators=FINAL_XGB_ESTIMATORS,
    max_depth=FINAL_XGB_MAX_DEPTH,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    eval_metric="logloss"
)
xgb_final_optimized.fit(X_train_hybrid, train_labels_arr)

print("Final optimized XGBoost leaf-generator trained.")
print(f"Config: n_estimators={FINAL_XGB_ESTIMATORS}, max_depth={FINAL_XGB_MAX_DEPTH}")

## Section 65: Final DeepTrace — Step 2: Leaf Features

In [ ]:
train_leaf_final = xgb_final_optimized.apply(X_train_hybrid)
test_leaf_final = xgb_final_optimized.apply(X_test_hybrid)

X_train_final_opt = np.concatenate([X_train_hybrid, train_leaf_final], axis=1)
X_test_final_opt = np.concatenate([X_test_hybrid, test_leaf_final], axis=1)

print("X_train_final_opt shape:", X_train_final_opt.shape)
print("X_test_final_opt shape :", X_test_final_opt.shape)

np.save("X_train_final_deeptrace_tuned.npy", X_train_final_opt)
np.save("X_test_final_deeptrace_tuned.npy", X_test_final_opt)

## Section 66: Final DeepTrace — Step 3: Final Random Forest

In [ ]:
rf_final_optimized = RandomForestClassifier(
    n_estimators=FINAL_RF_ESTIMATORS,
    random_state=RANDOM_SEED,
    n_jobs=-1
)
rf_final_optimized.fit(X_train_final_opt, train_labels_arr)

print("Final optimized Random Forest trained.")

## Section 67: Final DeepTrace — Step 4: Evaluation

In [ ]:
final_opt_preds = rf_final_optimized.predict(X_test_final_opt)
final_opt_probs = rf_final_optimized.predict_proba(X_test_final_opt)[:, 1]

final_opt_metrics = {
    "Accuracy": accuracy_score(test_labels_arr, final_opt_preds),
    "Precision": precision_score(test_labels_arr, final_opt_preds),
    "Recall": recall_score(test_labels_arr, final_opt_preds),
    "F1": f1_score(test_labels_arr, final_opt_preds)
}
print("DeepTrace (Final Tuned Model):", final_opt_metrics)
print(classification_report(test_labels_arr, final_opt_preds, target_names=["Safe","Unsafe"]))

cm_final_opt = confusion_matrix(test_labels_arr, final_opt_preds)
print(cm_final_opt)

fpr_final, tpr_final, _ = roc_curve(test_labels_arr, final_opt_probs)
roc_auc_final = auc(fpr_final, tpr_final)

prec_final, rec_final, _ = precision_recall_curve(test_labels_arr, final_opt_probs)
avg_prec_final = average_precision_score(test_labels_arr, final_opt_probs)

print("DeepTrace (Final Tuned) ROC-AUC:", roc_auc_final)
print("DeepTrace (Final Tuned) Average Precision:", avg_prec_final)

np.save("deeptrace_final_predictions.npy", final_opt_preds)
np.save("deeptrace_final_probs.npy", final_opt_probs)

## Section 68: Figure — DeepTrace Final Confusion Matrix; Save Model Artifacts

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(cm_final_opt, annot=True, fmt="d", cmap="Purples",
            xticklabels=["Safe","Unsafe"], yticklabels=["Safe","Unsafe"])
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("DeepTrace (Final Tuned Model) - Confusion Matrix")
plt.tight_layout()
plt.savefig("Figure26_DeepTrace_Final_Confusion_Matrix.png", dpi=300)
plt.show()

import joblib
joblib.dump(xgb_final_optimized, "xgb_final_optimized_model.joblib")
joblib.dump(rf_final_optimized, "rf_final_optimized_model.joblib")

print("Final DeepTrace model artifacts saved:")
print(" - xgb_final_optimized_model.joblib")
print(" - rf_final_optimized_model.joblib")

## Section 69: Master Model Comparison Table (6 Models)

In [ ]:
comparison_data = [
    {"Model": "BERT",
     "Accuracy": eval_results["eval_accuracy"], "Precision": eval_results["eval_precision"],
     "Recall": eval_results["eval_recall"], "F1": eval_results["eval_f1"]},

    {"Model": "BERT + XGBoost",
     "Accuracy": xgb_bert_metrics["Accuracy"], "Precision": xgb_bert_metrics["Precision"],
     "Recall": xgb_bert_metrics["Recall"], "F1": xgb_bert_metrics["F1"]},

    {"Model": "BERT + Random Forest",
     "Accuracy": rf_bert_metrics["Accuracy"], "Precision": rf_bert_metrics["Precision"],
     "Recall": rf_bert_metrics["Recall"], "F1": rf_bert_metrics["F1"]},

    {"Model": "Hybrid + XGBoost",
     "Accuracy": hybrid_xgb_metrics["Accuracy"], "Precision": hybrid_xgb_metrics["Precision"],
     "Recall": hybrid_xgb_metrics["Recall"], "F1": hybrid_xgb_metrics["F1"]},

    {"Model": "Hybrid + Random Forest",
     "Accuracy": hybrid_rf_metrics["Accuracy"], "Precision": hybrid_rf_metrics["Precision"],
     "Recall": hybrid_rf_metrics["Recall"], "F1": hybrid_rf_metrics["F1"]},

    {"Model": "DeepTrace (Proposed, Tuned)",
     "Accuracy": final_opt_metrics["Accuracy"], "Precision": final_opt_metrics["Precision"],
     "Recall": final_opt_metrics["Recall"], "F1": final_opt_metrics["F1"]},
]

master_comparison_df = pd.DataFrame(comparison_data).round(4)

master_comparison_df = master_comparison_df.sort_values("F1", ascending=False).reset_index(drop=True)
master_comparison_df.insert(0, "Rank", range(1, len(master_comparison_df) + 1))
master_comparison_df.to_csv("Master_Model_Comparison.csv", index=False)

print(master_comparison_df.to_string(index=False))

## Section 70: Total Misclassifications Comparison (Fewest to Most)

In [ ]:
error_counts = [
    {"Model": "BERT", "Errors": int((y_pred != y_true).sum())},
    {"Model": "BERT + XGBoost", "Errors": int((xgb_bert_preds != test_labels_arr).sum())},
    {"Model": "BERT + Random Forest", "Errors": int((rf_bert_preds != test_labels_arr).sum())},
    {"Model": "Hybrid + XGBoost", "Errors": int((hybrid_xgb_preds != test_labels_arr).sum())},
    {"Model": "Hybrid + Random Forest", "Errors": int((hybrid_rf_preds != test_labels_arr).sum())},
    {"Model": "DeepTrace (Proposed, Tuned)", "Errors": int((final_opt_preds != test_labels_arr).sum())},
]

error_counts_df = pd.DataFrame(error_counts)
error_counts_df = error_counts_df.sort_values("Errors", ascending=True).reset_index(drop=True)
error_counts_df.insert(0, "Rank", range(1, len(error_counts_df) + 1))
error_counts_df["Errors vs Best"] = error_counts_df["Errors"] - error_counts_df["Errors"].min()
error_counts_df.to_csv("Model_Error_Counts.csv", index=False)

print(error_counts_df.to_string(index=False))

## Section 71: Figure — Accuracy Comparison (High to Low)

In [ ]:
plt.figure(figsize=(11,6))
bars = plt.bar(master_comparison_df["Model"], master_comparison_df["Accuracy"], color="steelblue")
plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.title("Accuracy Comparison Across All Models (High to Low)")
plt.xticks(rotation=25, ha="right")
plt.ylim(master_comparison_df["Accuracy"].min() - 0.002, master_comparison_df["Accuracy"].max() + 0.002)

for bar, val in zip(bars, master_comparison_df["Accuracy"]):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.0002, f"{val:.4f}", ha="center", fontsize=9)

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("Figure27_Master_Accuracy_Comparison.png", dpi=300)
plt.show()

## Section 72: Figure — F1 Score Comparison (High to Low)

In [ ]:
plt.figure(figsize=(11,6))
bars = plt.bar(master_comparison_df["Model"], master_comparison_df["F1"], color="darkorange")
plt.ylabel("F1 Score")
plt.xlabel("Model")
plt.title("F1-Score Comparison Across All Models (High to Low)")
plt.xticks(rotation=25, ha="right")
plt.ylim(master_comparison_df["F1"].min() - 0.002, master_comparison_df["F1"].max() + 0.002)

for bar, val in zip(bars, master_comparison_df["F1"]):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.0002, f"{val:.4f}", ha="center", fontsize=9)

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("Figure28_Master_F1_Comparison.png", dpi=300)
plt.show()

## Section 73: Figure — ROC Curve Comparison (All Models, With Zoom Inset)

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

roc_models = [
    (xgb_bert_probs, "BERT+XGBoost", "tab:blue", "-"),
    (rf_bert_probs, "BERT+RF", "tab:orange", "--"),
    (hybrid_xgb_probs, "Hybrid+XGBoost", "tab:green", "-."),
    (hybrid_rf_probs, "Hybrid+RF", "tab:red", ":"),
    (final_opt_probs, "DeepTrace (Tuned)", "black", "-"),
]

fig, ax = plt.subplots(figsize=(9,9))

for probs, label, color, style in roc_models:
    fpr, tpr, _ = roc_curve(test_labels_arr, probs)
    roc_auc_val = auc(fpr, tpr)
    linewidth = 2.8 if label == "DeepTrace (Tuned)" else 1.8
    ax.plot(fpr, tpr, linewidth=linewidth, linestyle=style, color=color,
            label=f"{label} (AUC={roc_auc_val:.4f})")

ax.plot([0,1],[0,1], linestyle=":", color="gray", linewidth=1, label="Random Baseline")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curve Comparison - All Models", fontsize=13)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)

axins = inset_axes(ax, width="45%", height="45%", loc="center right",
                    bbox_to_anchor=(-0.05, -0.05, 1, 1), bbox_transform=ax.transAxes)

for probs, label, color, style in roc_models:
    fpr, tpr, _ = roc_curve(test_labels_arr, probs)
    linewidth = 2.8 if label == "DeepTrace (Tuned)" else 1.8
    axins.plot(fpr, tpr, linewidth=linewidth, linestyle=style, color=color)

axins.set_xlim(0.0, 0.08)
axins.set_ylim(0.92, 1.001)
axins.grid(alpha=0.3)
axins.set_title("Zoomed View", fontsize=9)
axins.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig("Figure29_Master_ROC_Comparison.png", dpi=300)
plt.show()

## Section 74: Figure — Precision/Recall/F1 Grouped Comparison (High to Low)

In [ ]:
x = np.arange(len(master_comparison_df))
width = 0.25

plt.figure(figsize=(12,6))
plt.bar(x - width, master_comparison_df["Precision"], width, label="Precision")
plt.bar(x, master_comparison_df["Recall"], width, label="Recall")
plt.bar(x + width, master_comparison_df["F1"], width, label="F1 Score")

plt.xticks(x, master_comparison_df["Model"], rotation=25, ha="right")
plt.ylabel("Score")
plt.title("Precision, Recall, and F1-Score Comparison - All Models (High to Low)")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.ylim(master_comparison_df[["Precision","Recall","F1"]].min().min() - 0.003,
          master_comparison_df[["Precision","Recall","F1"]].max().max() + 0.003)
plt.tight_layout()
plt.savefig("Figure30_Master_PRF_Comparison.png", dpi=300)
plt.show()

## Section 75: Figure — Total Misclassifications (Fewest to Most)

In [ ]:
plt.figure(figsize=(11,6))
bars = plt.bar(error_counts_df["Model"], error_counts_df["Errors"], color="indianred")
plt.ylabel("Number of Misclassified URLs (out of 10,000)")
plt.xlabel("Model")
plt.title("Total Misclassifications Across All Models (Fewest to Most)")
plt.xticks(rotation=25, ha="right")

for bar, val in zip(bars, error_counts_df["Errors"]):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.5, str(val), ha="center", fontsize=9)

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("Figure31_Master_Error_Comparison.png", dpi=300)
plt.show()

## Section 76: Build Error Analysis Dataframe

In [ ]:
error_df = pd.DataFrame()
error_df["url"] = test_texts.values
error_df["actual"] = test_labels_arr
error_df["deeptrace_pred"] = final_opt_preds
error_df["deeptrace_prob_unsafe"] = final_opt_probs

label_map = {0: "Safe", 1: "Unsafe"}
error_df["actual_label"] = error_df["actual"].map(label_map)
error_df["predicted_label"] = error_df["deeptrace_pred"].map(label_map)
error_df["is_correct"] = error_df["actual"] == error_df["deeptrace_pred"]

print("Total test samples:", len(error_df))
print("Correct predictions:", error_df["is_correct"].sum())
print("Incorrect predictions:", (~error_df["is_correct"]).sum())

## Section 77: False Positive / False Negative Breakdown

In [ ]:
false_positives = error_df[
    (error_df["actual"] == 0) & (error_df["deeptrace_pred"] == 1)
].copy()

false_negatives = error_df[
    (error_df["actual"] == 1) & (error_df["deeptrace_pred"] == 0)
].copy()

print("False Positives (Safe predicted as Unsafe):", len(false_positives))
print("False Negatives (Unsafe predicted as Safe):", len(false_negatives))

false_positives.to_csv("False_Positives.csv", index=False)
false_negatives.to_csv("False_Negatives.csv", index=False)

print("\nSample False Positives:")
print(false_positives[["url", "deeptrace_prob_unsafe"]].head(10))

print("\nSample False Negatives:")
print(false_negatives[["url", "deeptrace_prob_unsafe"]].head(10))

## Section 78: Feature Characteristics — Correct vs Misclassified

In [ ]:
url_feature_df = pd.DataFrame(X_test_url, columns=feature_names)
url_feature_df["is_correct"] = error_df["is_correct"].values

feature_summary = url_feature_df.groupby("is_correct")[feature_names].mean().T
feature_summary.columns = ["Misclassified", "Correct"]
feature_summary = feature_summary.round(3)

print(feature_summary)
feature_summary.to_csv("Error_Feature_Comparison.csv")

## Section 79: Figure — Feature Comparison (Correct vs Misclassified)

In [ ]:
key_features = ["url_length", "entropy", "digit_count", "dot_count",
                "hyphen_count", "login_keyword", "verify_keyword", "https_flag"]

plot_df = feature_summary.loc[key_features]

plot_df.plot(kind="bar", figsize=(12,6))
plt.ylabel("Average Value")
plt.title("Feature Characteristics - Correct vs Misclassified Predictions")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("Figure32_Error_Feature_Comparison.png", dpi=300)
plt.show()

## Section 80: Feature Importance — Final Tuned Random Forest (By Group)

In [ ]:
importances = rf_final_optimized.feature_importances_

bert_importance = importances[:768].sum()
url_importance = importances[768:788].sum()
leaf_importance = importances[788:].sum()

importance_group_df = pd.DataFrame({
    "Feature Group": ["BERT Embeddings (768)", "Handcrafted URL Features (20)", "XGBoost Leaf Indices (150)"],
    "Total Importance": [bert_importance, url_importance, leaf_importance]
})
importance_group_df["Percentage"] = (
    importance_group_df["Total Importance"] / importance_group_df["Total Importance"].sum() * 100
).round(2)

print(importance_group_df)
importance_group_df.to_csv("Feature_Group_Importance.csv", index=False)

## Section 81: Figure — Feature Group Importance

In [ ]:
plt.figure(figsize=(8,6))
bars = plt.bar(importance_group_df["Feature Group"], importance_group_df["Total Importance"],
               color=["steelblue","seagreen","indianred"])
plt.ylabel("Cumulative Feature Importance")
plt.title("Contribution of Feature Groups - Final Tuned Random Forest (DeepTrace)")
plt.xticks(rotation=15, ha="right")

for bar, pct in zip(bars, importance_group_df["Percentage"]):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f"{pct}%", ha="center", fontsize=10)

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("Figure33_Feature_Group_Importance.png", dpi=300)
plt.show()

## Section 82: Top Handcrafted Feature Importances (Individual)

In [ ]:
url_feature_importances = importances[768:788]
url_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": url_feature_importances
}).sort_values("Importance", ascending=False)

print(url_importance_df)

plt.figure(figsize=(10,6))
plt.barh(url_importance_df["Feature"], url_importance_df["Importance"], color="teal")
plt.xlabel("Importance")
plt.title("Individual Handcrafted Feature Importance - Final Tuned Random Forest (DeepTrace)")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("Figure34_Handcrafted_Feature_Importance.png", dpi=300)
plt.show()

## End of Notebook

All results, figures, and model artifacts are saved and ready for export. See the accompanying dissertation report document for the full write-up of each section's methodology, results, and interpretation, referencing these exact section numbers for every figure and table.